# Task 6 — System Demonstration
## Multimodal Authentication & Product Recommendation Pipeline

**Team:** Data Preprocessing Group  
**Author:** Kelvin (Task 6 — Integration & Demonstration)

---

### What This Notebook Does
Connects all teammates' work into a single, running end-to-end demo:

| Stage | Model | Source |
|---|---|---|
| Face Recognition | `facial_recognition_model.pkl` | Task 4 |
| Product Recommendation | `product_recommendation_model.pkl` | Task 4 |
| Voice Verification | `voiceprint_model.pkl` | Task 4 |

### Pipeline (from team flowchart)
```
User Input
    ↓
[STEP 1] Face Recognition  →  FAIL → Access Denied
    ↓ PASS
[STEP 2] Product Recommendation  (computed, held until voice passes)
    ↓
[STEP 3] Voice Verification  →  FAIL → Access Denied
    ↓ PASS
Display: Welcome + Recommended Product
```


## Cell 1 — Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import joblib
import os

warnings.filterwarnings('ignore')
print('Libraries loaded.')


## Cell 2 — Paths & Member→Customer ID Mapping

In [ ]:
# ── Directory paths ────────────────────────────────────────────────────
MODELS_DIR   = 'saved_models'
DATA_DIR     = os.path.join('data', 'processed')
FEATURES_DIR = 'features'

IMAGE_CSV  = os.path.join(DATA_DIR,     'image_features.csv')
AUDIO_CSV  = os.path.join(FEATURES_DIR, 'audio_features.csv')
MERGED_CSV = os.path.join(DATA_DIR,     'merged_dataset.csv')

# ── Member → Customer ID ────────────────────────────────────────────────
# Each team member is manually mapped to a real customer ID in the dataset.
# This is how biometric identity connects to purchase history.
#
# NOTE: A178 (originally assigned to Kelvin) was not present in the
# merged dataset, so A190 is used for Kelvin instead.
MEMBER_TO_CUSTOMER_ID = {
    'David'         : 'A192',
    'Kelvin'        : 'A190',
    'Michael Kimani': 'A150',
    'Samuel'        : 'A103',
}

print('Member → Customer ID mapping:')
for member, cid in MEMBER_TO_CUSTOMER_ID.items():
    print(f'  {member:20s} → {cid}')


## Cell 3 — Load All Models

In [ ]:
face_model   = joblib.load(os.path.join(MODELS_DIR, 'facial_recognition_model.pkl'))
face_enc     = joblib.load(os.path.join(MODELS_DIR, 'face_label_encoder.pkl'))

voice_model  = joblib.load(os.path.join(MODELS_DIR, 'voiceprint_model.pkl'))
voice_enc    = joblib.load(os.path.join(MODELS_DIR, 'voice_label_encoder.pkl'))
audio_scaler = joblib.load(os.path.join(MODELS_DIR, 'audio_scaler.pkl'))

prod_model   = joblib.load(os.path.join(MODELS_DIR, 'product_recommendation_model.pkl'))
prod_enc     = joblib.load(os.path.join(MODELS_DIR, 'product_label_encoder.pkl'))
prod_cols    = joblib.load(os.path.join(MODELS_DIR, 'product_feature_columns.pkl'))

print('All models loaded successfully.')
print(f'  Face model classes   : {list(face_enc.classes_)}')
print(f'  Voice model classes  : {list(voice_enc.classes_)}')
print(f'  Product categories   : {list(prod_enc.classes_)}')


## Cell 4 — Load Feature Data

In [ ]:
# ── Image features (Task 2 output) ─────────────────────────────────────
image_df = pd.read_csv(IMAGE_CSV)
print(f'Image features  : {image_df.shape}')

# ── Merged dataset (Task 1 output) ──────────────────────────────────────
merged_df = pd.read_csv(MERGED_CSV)
bool_cols = merged_df.select_dtypes(include='bool').columns
merged_df[bool_cols] = merged_df[bool_cols].astype(int)
print(f'Merged dataset  : {merged_df.shape}')

# ── Audio features (Task 3 output) ──────────────────────────────────────
# When your teammate provides audio_features.csv, this block will auto-detect it.
if os.path.exists(AUDIO_CSV):
    audio_df = pd.read_csv(AUDIO_CSV)
    print(f'Audio features  : {audio_df.shape}  [REAL DATA from Task 3]')
else:
    print('NOTE: audio_features.csv not found in features/ folder.')
    print('      Using synthetic audio features for demo purposes.')
    print('      To use real data: place audio_features.csv in the features/ folder.')
    
    AUDIO_COLS = [f'mfcc_{i}' for i in range(1, 14)] + ['spectral_rolloff', 'rms_energy', 'zcr']
    MEMBERS    = list(MEMBER_TO_CUSTOMER_ID.keys())
    np.random.seed(42)
    member_centers = {m: np.random.randn(len(AUDIO_COLS)) * 10 for m in MEMBERS}
    
    rows = []
    for member in MEMBERS:
        center = member_centers[member]
        for phrase in ['yes_approve', 'confirm_transaction']:
            for stype in ['original', 'augmented']:
                noise = np.random.randn(len(AUDIO_COLS)) * 0.5
                row = {'member': member, 'phrase_label': phrase,
                       'sample_type': stype, 'file_name': f'{member}_{phrase}_{stype}.wav'}
                for j, col in enumerate(AUDIO_COLS):
                    row[col] = center[j] + noise[j]
                rows.append(row)
            for k in range(4):
                noise = np.random.randn(len(AUDIO_COLS)) * 0.5
                row = {'member': member, 'phrase_label': phrase,
                       'sample_type': 'augmented', 'file_name': f'{member}_{phrase}_aug{k}.wav'}
                for j, col in enumerate(AUDIO_COLS):
                    row[col] = center[j] + noise[j]
                rows.append(row)
    audio_df = pd.DataFrame(rows)
    print(f'Synthetic audio features shape: {audio_df.shape}')

# ── Feature column sets ──────────────────────────────────────────────────
FACE_FEATURES  = [c for c in image_df.columns if c not in ['member', 'expression', 'augmentation']]
AUDIO_FEATURES = [c for c in audio_df.columns if c not in ['member', 'phrase_label', 'sample_type', 'file_name']]

print(f'\nFace feature columns  : {len(FACE_FEATURES)}')
print(f'Audio feature columns : {len(AUDIO_FEATURES)}')
print(f'Product feature cols  : {len(prod_cols)}')


## Cell 5 — Customer Profile Helper

In [ ]:
def get_customer_profile(customer_id: str):
    """
    Fetches the most recent transaction row for a given customer ID.
    Returns the row as a dict, or None if the customer ID is not found.
    """
    subset = merged_df[merged_df['customer_id_new'] == customer_id]
    if subset.empty:
        return None
    return subset.sort_values('purchase_month', ascending=False).iloc[0].to_dict()


# ── Verify all 4 members have valid profiles ────────────────────────────
print('Customer profile verification:')
print(f'{"Member":<20} {"Customer ID":<12} {"Last Category":<15} {"Amount":>8}  Status')
print('-' * 65)
for member, cid in MEMBER_TO_CUSTOMER_ID.items():
    profile = get_customer_profile(cid)
    if profile:
        print(f'{member:<20} {cid:<12} {profile["product_category"]:<15} ${profile["purchase_amount"]:>7.2f}  ✅')
    else:
        print(f'{member:<20} {cid:<12} {"N/A":<15} {"N/A":>9}  ❌ NOT FOUND')


## Cell 6 — Core Pipeline Function

In [ ]:
def run_pipeline(face_member: str, audio_member: str, label: str = 'Simulation'):
    """
    Runs the full 3-stage authentication + recommendation pipeline.
    
    Parameters
    ----------
    face_member  : str  Name of the member whose face features to use
    audio_member : str  Name of the member whose voice features to use
    label        : str  Description label for this simulation run
    
    Stage order (matches team flowchart):
      1. Face Recognition
      2. Product Recommendation  (runs after face, displayed after voice)
      3. Voice Verification
    """
    print(f'\n{"=" * 60}')
    print(f'  {label}')
    print(f'{"=" * 60}')

    # ── STAGE 1: Face Recognition ─────────────────────────────────────────
    print('\n[STAGE 1]  FACE RECOGNITION')
    face_row   = image_df[
        (image_df['member'] == face_member) &
        (image_df['augmentation'] == 'original')
    ].iloc[0]
    face_input = face_row[FACE_FEATURES].values.reshape(1, -1)
    face_pred  = face_model.predict(face_input)[0]
    face_proba = face_model.predict_proba(face_input).max()
    face_name  = face_enc.inverse_transform([face_pred])[0]

    print(f'  Identified as : {face_name}')
    print(f'  Confidence    : {face_proba:.1%}')

    if face_proba < 0.50:
        print('  ❌  ACCESS DENIED — face not recognized.')
        print(f'{"=" * 60}\n')
        return

    print('  ✅  Face recognized. Moving to product recommendation...')

    # ── STAGE 2: Product Recommendation ──────────────────────────────────
    print('\n[STAGE 2]  PRODUCT RECOMMENDATION')
    customer_id = MEMBER_TO_CUSTOMER_ID.get(face_name)
    profile     = get_customer_profile(customer_id)

    if profile is None:
        print(f'  ❌  No profile found for customer {customer_id}.')
        print(f'{"=" * 60}\n')
        return

    prod_input  = pd.DataFrame([profile])[prod_cols].values.astype(float)
    prod_pred   = prod_model.predict(prod_input)[0]
    prod_proba  = prod_model.predict_proba(prod_input).max()
    recommended = prod_enc.inverse_transform([prod_pred])[0]

    print(f'  Customer ID        : {customer_id}')
    print(f'  Predicted category : {recommended}')
    print(f'  Confidence         : {prod_proba:.1%}')
    print('  ✅  Recommendation ready. Moving to voice verification...')

    # ── STAGE 3: Voice Verification ───────────────────────────────────────
    print('\n[STAGE 3]  VOICE VERIFICATION')
    audio_row   = audio_df[
        (audio_df['member'] == audio_member) &
        (audio_df['sample_type'] == 'original')
    ].iloc[0]
    audio_input = audio_scaler.transform(
        audio_row[AUDIO_FEATURES].values.reshape(1, -1)
    )
    voice_pred  = voice_model.predict(audio_input)[0]
    voice_proba = voice_model.predict_proba(audio_input).max()
    voice_name  = voice_enc.inverse_transform([voice_pred])[0]

    print(f'  Voice identified  : {voice_name}')
    print(f'  Confidence        : {voice_proba:.1%}')

    if voice_name != face_name:
        print(f'  ❌  ACCESS DENIED — voice ({voice_name}) does not match face ({face_name}).')
        print(f'{"=" * 60}\n')
        return

    print('  ✅  Voice verified.')

    # ── FINAL OUTPUT ──────────────────────────────────────────────────────
    print(f'\n  🎉  AUTHENTICATION SUCCESSFUL')
    print(f'  Welcome, {face_name}!  (Customer ID: {customer_id})')
    print(f'  Recommended product category: {recommended}')
    print(f'{"=" * 60}\n')


print('Pipeline function defined and ready.')


## Cell 7 — Simulation 1: Authorized Users (All 4 Members)

In [ ]:
print('\n' + '█' * 60)
print('  SIMULATION 1 — AUTHORIZED USERS')
print('  Each member uses their OWN face and their OWN voice')
print('█' * 60)

for member in ['David', 'Kelvin', 'Michael Kimani', 'Samuel']:
    run_pipeline(
        face_member  = member,
        audio_member = member,
        label        = f'Authorized User — {member}'
    )


## Cell 8 — Simulation 2: Unauthorized — Face/Voice Mismatch

In [ ]:
print('\n' + '█' * 60)
print('  SIMULATION 2 — UNAUTHORIZED ATTEMPTS')
print('  Attacker submits one person\'s face but a different voice')
print('█' * 60)

# Scenario A: Kelvin's face, Samuel's voice
run_pipeline(
    face_member  = 'Kelvin',
    audio_member = 'Samuel',
    label        = 'UNAUTHORIZED — Kelvin face + Samuel voice'
)

# Scenario B: David's face, Michael's voice
run_pipeline(
    face_member  = 'David',
    audio_member = 'Michael Kimani',
    label        = 'UNAUTHORIZED — David face + Michael Kimani voice'
)

# Scenario C: Michael's face, David's voice
run_pipeline(
    face_member  = 'Michael Kimani',
    audio_member = 'David',
    label        = 'UNAUTHORIZED — Michael Kimani face + David voice'
)


## Cell 9 — Simulation 3: Unknown Face (Low Confidence)

In [ ]:
print('\n' + '█' * 60)
print('  SIMULATION 3 — UNKNOWN USER (STRANGER)')
print('  Random features that do not belong to any registered member')
print('█' * 60)

np.random.seed(99)
stranger_features = np.random.uniform(
    low  = image_df[FACE_FEATURES].min().values,
    high = image_df[FACE_FEATURES].max().values
)

face_input = stranger_features.reshape(1, -1)
face_pred  = face_model.predict(face_input)[0]
face_proba = face_model.predict_proba(face_input).max()
face_name  = face_enc.inverse_transform([face_pred])[0]

print(f'\n{"=" * 60}')
print(f'  UNAUTHORIZED — Unknown Stranger')
print(f'{"=" * 60}')
print(f'\n[STAGE 1]  FACE RECOGNITION')
print(f'  Best guess : {face_name}')
print(f'  Confidence : {face_proba:.1%}')

THRESHOLD = 0.60
if face_proba < THRESHOLD:
    print(f'  ❌  ACCESS DENIED — confidence {face_proba:.1%} is below threshold ({THRESHOLD:.0%}).')
    print('      Face not recognized as any registered member.')
else:
    print(f'  ✅  Recognized as {face_name} (this means stranger happened to match a member)')
print(f'{"=" * 60}\n')


## Cell 10 — Summary Report

In [ ]:
print('\n' + '=' * 70)
print('  SYSTEM DEMONSTRATION SUMMARY')
print('=' * 70)

scenarios = [
    ('Authorized — David',                'David',          'David',          'APPROVED ✅'),
    ('Authorized — Kelvin',               'Kelvin',         'Kelvin',         'APPROVED ✅'),
    ('Authorized — Michael Kimani',       'Michael Kimani', 'Michael Kimani', 'APPROVED ✅'),
    ('Authorized — Samuel',               'Samuel',         'Samuel',         'APPROVED ✅'),
    ('Unauth — Kelvin face + Samuel voice','Kelvin',        'Samuel',         'DENIED  ❌'),
    ('Unauth — David face + Michael voice','David',         'Michael Kimani', 'DENIED  ❌'),
    ('Unauth — Michael face + David voice','Michael Kimani','David',          'DENIED  ❌'),
    ('Unauth — Unknown stranger',          'N/A',           'N/A',            'DENIED  ❌'),
]

print(f'  {"Scenario":<45} {"Face":<16} {"Voice":<16} {"Result"}')
print('  ' + '-' * 66)
for scenario, face, voice, result in scenarios:
    print(f'  {scenario:<45} {face:<16} {voice:<16} {result}')

print('=' * 70)
print('''
ARCHITECTURE SUMMARY
──────────────────────────────────────────────────────────────────
  Models:
    facial_recognition_model.pkl   — Random Forest, 4 members
    voiceprint_model.pkl           — Random Forest, 4 members  
    product_recommendation_model.pkl — Random Forest, 5 categories

  Data sources:
    image_features.csv    → Face recognition training (Task 2)
    audio_features.csv    → Voice verification training (Task 3)
    merged_dataset.csv    → Product recommendation training (Task 1)

  Identity bridge:
    Face model  → returns member name
    Member name → maps to customer ID (MEMBER_TO_CUSTOMER_ID dict)
    Customer ID → fetches purchase history from merged_dataset
    Purchase history → fed to product recommendation model

  Security gates:
    Gate 1 (Face):  confidence < 50%  → Access Denied
    Gate 2 (Voice): voice name ≠ face name → Access Denied
──────────────────────────────────────────────────────────────────
''')
